# KrishiNexus · `krishi-price-v1`: staple-price forecasts and market-margin anomalies

Forecasts monthly retail prices for coarse rice, wheat flour, masur lentils and palm oil in six divisions of
Bangladesh, 1, 2 and 3 months ahead. It also flags months when Dhaka's retail margin over wholesale is unusual.

**Data.** WFP, *Bangladesh - Food Prices*, Humanitarian Data Exchange, licence CC BY-IGO.
- The snapshot was last modified on 2026-11-04 and is attached as the private dataset `krishinexus-bd-food-prices`.
- WFP reported division averages until mid-2020 and district markets after. Each series joins the two, and no
  feature or target is computed across the join.

**Models:** naive, seasonal naive, LightGBM (one per horizon), a GRU, a TCN, and the mean of the three learned
models.

**Features:**
- past log-returns of the price;
- the month, and the share of it that falls in Ramadan;
- the taka-dollar rate implied by WFP's own conversions;
- monthly rainfall anomaly from the Open-Meteo archive.

**Frozen-test rule:**
- train: targets before 2024-10;
- validation: 2024-10 to 2025-09, used to choose the model;
- test: 2025-10 to 2026-09, scored once, here.

The targets were set before any training (TRAINING_PLAN.md section 8b):
- mean sMAPE below the naive forecast's;
- 3-month sMAPE below the seasonal-naive forecast's;
- 3-month direction accuracy ≥ 0.55.

**Spread anomalies:** Dhaka coarse rice, medium rice and wheat flour, 2008–2021. A month is flagged by a trailing
robust z-score of 3.5 or more, or by an IsolationForest. The flags are compared with three publicly reported
disruption windows. This is a case study, not a metric.

**Needs:** CPU only. Internet on, for the rainfall archive; without it the rainfall feature is dropped.

**Outputs (`/kaggle/working/krishi-price-v1`):** `price_eval_report.json`, `price_forecasts.json`, `lightgbm_h*.txt`,
`gru.pt` and `tcn.pt`.


In [ ]:
import glob, sys
import lightgbm, numpy, pandas, sklearn, torch
print({m.__name__: m.__version__ for m in (lightgbm, numpy, pandas, sklearn, torch)}, "| python", sys.version.split()[0])


In [ ]:
"""KrishiNexus price intelligence (krishi-price-v1): staple-price forecasts and wholesale-retail margin anomalies.

Built from WFP's "Bangladesh - Food Prices" on HDX. TRAINING_PLAN.md section 8b is the design and holds the targets,
which were set before any training. The Kaggle notebook embeds this file verbatim and calls run().
"""
import json
import urllib.request

import numpy as np
import pandas as pd

STAPLES = {"Rice (coarse)": "rice_coarse", "Wheat flour": "wheat_flour", "Lentils (masur)": "lentils_masur",
           "Oil (palm)": "palm_oil"}
DIVISIONS = ("Barisal", "Chittagong", "Dhaka", "Khulna", "Rajshahi", "Sylhet")   # WFP division series since 2004-07
HORIZONS = (1, 2, 3)
VAL_START, TEST_START, TEST_END = pd.Period("2024-10", "M"), pd.Period("2025-10", "M"), pd.Period("2026-09", "M")
WINDOW = 24                                            # months of history the sequence models read
SPREAD_PAIRS = {"Rice (coarse, BR-8/ 11/, Guti Sharna)": "rice_coarse", "Rice (medium grain)": "rice_medium",
                "Wheat flour": "wheat_flour"}          # Dhaka only: the one market with wholesale and retail together
EVENTS = {"2008 food-price crisis": ("2008-01", "2008-06"), "2017 haor-flood rice spike": ("2017-04", "2017-09"),
          "2020 COVID-19 lockdown": ("2020-03", "2020-06")}
SEED = 20261109


# ---------------------------------------------------------------------------------------------------------- helpers

def islamic_month(y, m, d):
    """Month (1-12) of the tabular Islamic calendar, within a day or two of the observed one."""
    a = (14 - m) // 12
    yy, mm = y + 4800 - a, m + 12 * a - 3
    jdn = d + (153 * mm + 2) // 5 + 365 * yy + yy // 4 - yy // 100 + yy // 400 - 32045
    l = jdn - 1948440 + 10632
    n = (l - 1) // 10631
    l = l - 10631 * n + 354
    j = ((10985 - l) // 5316) * ((50 * l) // 17719) + (l // 5670) * ((43 * l) // 15238)
    l = l - ((30 - j) // 15) * ((17719 * j) // 50) - (j // 16) * ((15238 * j) // 43) + 29
    return (24 * l) // 709


def ramadan_share(period):
    """Share of a calendar month's days that fall in Ramadan (month 9)."""
    days = pd.date_range(period.start_time, period.end_time.normalize(), freq="D")
    return float(np.mean([islamic_month(t.year, t.month, t.day) == 9 for t in days]))


def smape(actual, forecast):
    a, f = np.asarray(actual, float), np.asarray(forecast, float)
    return float(np.mean(2 * np.abs(f - a) / (np.abs(a) + np.abs(f))) * 100)


def direction_accuracy(last, actual, forecast, min_move=0.01):
    """Share of real moves larger than `min_move` whose direction the forecast called right."""
    l, a, f = (np.asarray(x, float) for x in (last, actual, forecast))
    moved = np.abs(a / l - 1) > min_move
    return round(float(np.mean(np.sign(a[moved] - l[moved]) == np.sign(f[moved] - l[moved]))), 4) if moved.any() else None


def robust_z(values, window=24, min_periods=12):
    """Trailing robust z-score: distance from the previous `window` values' median, in 1.4826 x MAD units."""
    v = np.asarray(values, float)
    out = np.full(len(v), np.nan)
    for i in range(len(v)):
        hist = v[max(0, i - window):i]
        hist = hist[~np.isnan(hist)]
        if len(hist) >= min_periods and not np.isnan(v[i]):
            med = np.median(hist)
            mad = 1.4826 * np.median(np.abs(hist - med)) or 1.2533 * np.mean(np.abs(hist - med))   # MAD 0: mean deviation
            out[i] = (v[i] - med) / mad if mad > 0 else (0.0 if v[i] == med else np.sign(v[i] - med) * np.inf)
    return out


def in_events(period):
    return any(pd.Period(a, "M") <= period <= pd.Period(b, "M") for a, b in EVENTS.values())


# ------------------------------------------------------------------------------------------------------------- data

def load_wfp(path):
    df = pd.read_csv(path, skiprows=[1])                 # row 2 is HDX's HXL tag row
    df["month"] = pd.PeriodIndex(df["date"], freq="M")
    for col in ("price", "usdprice", "latitude", "longitude"):
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def staple_series(df):
    """{(staple, division): monthly retail price} and the log-jump at each 2020 join.

    WFP reported division averages until mid-2020 and district markets after, so each series is the division average
    up to its last month, then the median of the division's district markets."""
    r = df[(df.pricetype == "Retail") & (df.currency == "BDT") & df.commodity.isin(list(STAPLES)) & (df.price > 0)]
    is_div = r.market.str.endswith(" Division")
    old = r[is_div].assign(division=r[is_div].market.str.removesuffix(" Division")).groupby(
        ["commodity", "division", "month"]).price.median()
    new = r[~is_div].groupby(["commodity", "admin1", "month"]).price.median()
    series, joins = {}, {}
    for c in STAPLES:
        for d in DIVISIONS:
            try:
                o = old.loc[(c, d)]
            except KeyError:
                continue
            n = new.loc[(c, d)] if (c, d) in new.index.droplevel(2) else pd.Series(dtype=float)
            after = n[n.index > o.index.max()]
            s = pd.concat([o, after]).sort_index()
            s = s.reindex(pd.period_range(s.index.min(), s.index.max(), freq="M"))
            if len(after):
                joins[f"{STAPLES[c]}|{d}"] = {"first_month": after.index.min(), "log_jump": round(float(
                    np.log(after.iloc[:3].mean() / o.iloc[-3:].mean())), 4)}
            series[(c, d)] = s
    return series, joins


def fx_series(df):
    """Taka per US dollar, implied by WFP's own conversions (price / usdprice), monthly median."""
    ok = df[(df.currency == "BDT") & (df.usdprice > 0) & (df.price > 0)]
    return (ok.price / ok.usdprice).groupby(ok.month).median()


def rain_anomaly(df, end):
    """{division: monthly rainfall minus that month's mean before VAL_START} from the Open-Meteo archive, or {}."""
    out = {}
    for d in DIVISIONS:
        lat, lon = df.loc[df.market == f"{d} Division", ["latitude", "longitude"]].iloc[0]
        url = ("https://archive-api.open-meteo.com/v1/archive?latitude=%.4f&longitude=%.4f&start_date=2003-01-01"
               "&end_date=%s&daily=precipitation_sum&timezone=Asia%%2FDhaka" % (lat, lon, end))
        try:
            with urllib.request.urlopen(url, timeout=120) as resp:
                daily = json.loads(resp.read())["daily"]
        except Exception as error:                     # rainfall is optional: the models run without it
            print(f"rainfall for {d} unavailable ({error}); the feature is dropped")
            return {}
        mm = pd.Series(daily["precipitation_sum"], index=pd.PeriodIndex(daily["time"], freq="M"), dtype=float)
        monthly = mm.groupby(level=0).sum(min_count=20)
        clim = monthly[monthly.index < VAL_START].groupby(monthly[monthly.index < VAL_START].index.month).mean()
        out[d] = monthly - clim.reindex(monthly.index.month).to_numpy()
    return out


# --------------------------------------------------------------------------------------------------------- features

_RAMADAN = {}


def ramadan_cached(period):
    if period not in _RAMADAN:
        _RAMADAN[period] = ramadan_share(period)
    return _RAMADAN[period]


def build_frame(series, joins, fx, rain):
    """One row per (series, origin month): features known at the origin and the log-return to each horizon.

    Each series is two segments split at its 2020 join, and no lag, rolling statistic or target crosses the join."""
    fx_log = np.log(fx)
    parts = []
    for (c, d), s in series.items():
        key = f"{STAPLES[c]}|{d}"
        f = pd.DataFrame({"y": np.log(s)}, index=s.index)
        f["seg"] = (f.index >= joins[key]["first_month"]).astype(int) if key in joins else 0
        g = f.groupby("seg")["y"]
        for k in (1, 2, 3, 6, 12):
            f[f"r{k}"] = f["y"] - g.shift(k)
        f["vol6"] = f.groupby("seg")["r1"].transform(lambda r: r.rolling(6, min_periods=3).std())
        f["dev12"] = f["y"] - g.transform(lambda v: v.rolling(12, min_periods=6).mean())
        f["fx_r1"] = fx_log.diff().reindex(f.index).to_numpy()
        f["fx_r3"] = fx_log.diff(3).reindex(f.index).to_numpy()
        r = rain.get(d)
        f["rain0"] = r.reindex(f.index).to_numpy() if r is not None else np.nan
        f["rain1"] = r.shift(1).reindex(f.index).to_numpy() if r is not None else np.nan
        f["staple"], f["division"], f["key"] = list(STAPLES).index(c), DIVISIONS.index(d), key
        f["ramadan0"] = [ramadan_cached(m) for m in f.index]
        for h in HORIZONS:
            f[f"t{h}"] = g.shift(-h) - f["y"]
        parts.append(f.reset_index(names="month"))
    return pd.concat(parts, ignore_index=True)


LGB_FEATURES = ["r1", "r2", "r3", "r6", "r12", "vol6", "dev12", "fx_r1", "fx_r3", "rain0", "rain1", "staple",
                "division", "ramadan0", "month_t", "ramadan_t"]


def split_of(target_month):
    if target_month < VAL_START:
        return "train"
    if target_month < TEST_START:
        return "val"
    return "test" if target_month <= TEST_END else "later"


def horizon_rows(frame, h):
    """Rows with a known origin and target h months ahead, labelled train / val / test by the target month."""
    x = frame[frame["y"].notna() & frame[f"t{h}"].notna()].copy()
    x["target_month"] = x["month"] + h
    x["month_t"] = [m.month for m in x["target_month"]]
    x["ramadan_t"] = [ramadan_cached(m) for m in x["target_month"]]
    x["split"] = [split_of(m) for m in x["target_month"]]
    x["target"] = x[f"t{h}"]
    return x


# ----------------------------------------------------------------------------------------------------------- models

def fit_lightgbm(x):
    import lightgbm as lgb
    params = dict(objective="l1", learning_rate=0.03, num_leaves=15, min_child_samples=20, feature_fraction=0.8,
                  bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, seed=SEED, deterministic=True, verbose=-1)
    tr, va = x[x["split"] == "train"], x[x["split"] == "val"]
    data = lgb.Dataset(tr[LGB_FEATURES], tr["target"], categorical_feature=["staple", "division"])
    valid = lgb.Dataset(va[LGB_FEATURES], va["target"], reference=data)
    return lgb.train(params, data, 3000, valid_sets=[valid], callbacks=[lgb.early_stopping(150, verbose=False)])


def sequence_arrays(frame, rain_on):
    """Per origin month: the last WINDOW months of inputs, the three horizon targets, and each target's split."""
    xs, ys, splits, index = [], [], [], []
    for key, f in frame.groupby("key", sort=False):
        f = f.sort_values("month")
        month_no = np.array([m.month for m in f["month"]])
        steps = np.column_stack([
            np.nan_to_num(f["r1"].to_numpy() * 10), f["y"].notna().to_numpy(float),
            np.sin(2 * np.pi * month_no / 12), np.cos(2 * np.pi * month_no / 12), f["ramadan0"].to_numpy(),
            np.nan_to_num(f["fx_r1"].to_numpy() * 10),
            np.nan_to_num(f["rain0"].to_numpy() / 100) if rain_on else np.zeros(len(f)),
        ])
        static = np.zeros(len(STAPLES) + len(DIVISIONS))
        static[int(f["staple"].iloc[0])] = 1
        static[len(STAPLES) + int(f["division"].iloc[0])] = 1
        steps = np.hstack([steps, np.tile(static, (len(f), 1))])
        months, y = f["month"].to_numpy(), f["y"].to_numpy()
        targets = f[[f"t{h}" for h in HORIZONS]].to_numpy(float)
        for i in range(WINDOW - 1, len(f)):
            if np.isnan(y[i]):
                continue
            xs.append(steps[i - WINDOW + 1:i + 1])
            ys.append(targets[i] * 10)
            splits.append([split_of(months[i] + h) for h in HORIZONS])
            index.append((key, months[i]))
    return np.array(xs, np.float32), np.array(ys, np.float32), np.array(splits), index


def fit_sequence(kind, xs, ys, splits, max_epochs=300, patience=30):
    """A GRU or a causal TCN over the input window, trained with masked L1 and early-stopped on validation loss."""
    import torch
    from torch import nn

    torch.manual_seed(SEED)
    n_in = xs.shape[2]

    class Net(nn.Module):
        def __init__(self):
            super().__init__()
            if kind == "gru":
                self.body = nn.GRU(n_in, 32, batch_first=True)
            else:
                layers, c = [], n_in
                for dilation in (1, 2, 4, 8):          # causal: each output sees only its own and earlier months
                    layers += [nn.ConstantPad1d((2 * dilation, 0), 0.0), nn.Conv1d(c, 32, 3, dilation=dilation),
                               nn.ReLU(), nn.Dropout(0.1)]
                    c = 32
                self.body = nn.Sequential(*layers)
            self.head = nn.Linear(32, len(HORIZONS))

        def forward(self, x):
            if kind == "gru":
                return self.head(self.body(x)[0][:, -1])
            return self.head(self.body(x.transpose(1, 2))[:, :, -1])

    net = Net()
    opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-4)
    x, y = torch.tensor(xs), torch.tensor(np.nan_to_num(ys))
    known = torch.tensor(~np.isnan(ys))
    masks = {s: known & torch.tensor(splits == s) for s in ("train", "val")}
    rows = {s: torch.nonzero(masks[s].any(1)).squeeze(1) for s in masks}
    gen = torch.Generator().manual_seed(SEED)

    def loss(sel, mask):
        return (torch.abs(net(x[sel]) - y[sel]) * mask[sel]).sum() / mask[sel].sum().clamp(min=1)

    best, best_state, waited, epochs = float("inf"), None, 0, 0
    for epochs in range(1, max_epochs + 1):
        net.train()
        for batch in rows["train"][torch.randperm(len(rows["train"]), generator=gen)].split(128):
            opt.zero_grad()
            loss(batch, masks["train"]).backward()
            opt.step()
        net.eval()
        with torch.no_grad():
            v = float(loss(rows["val"], masks["val"]))
        if v < best - 1e-5:
            best, best_state, waited = v, {k: t.clone() for k, t in net.state_dict().items()}, 0
        else:
            waited += 1
            if waited >= patience:
                break
    net.load_state_dict(best_state)
    net.eval()
    net.epochs_run = epochs
    return net


def predict_sequence(net, xs):
    import torch
    with torch.no_grad():
        return net(torch.tensor(xs)).numpy() / 10


# ------------------------------------------------------------------------------------------------- spread anomalies

def spread_anomalies(df):
    """Dhaka retail margin over wholesale per month, flagged by a robust z-score and by an IsolationForest."""
    from sklearn.ensemble import IsolationForest

    flagged, stats = [], {}
    for commodity, key in SPREAD_PAIRS.items():
        x = df[(df.market == "Dhaka") & (df.commodity == commodity) & (df.currency == "BDT") & (df.price > 0)]
        retail = x[(x.pricetype == "Retail") & (x.unit == "KG")].groupby("month").price.median()
        wholesale = x[(x.pricetype == "Wholesale") & (x.unit == "100 KG")].groupby("month").price.median() / 100
        m = pd.concat([retail.rename("retail"), wholesale.rename("wholesale")], axis=1).dropna()
        m = m.reindex(pd.period_range(m.index.min(), m.index.max(), freq="M"))
        m["margin"] = m["retail"] / m["wholesale"] - 1
        m["z"] = robust_z(m["margin"].to_numpy())
        feats = pd.DataFrame({"margin": m["margin"], "d_margin": m["margin"].diff(),
                              "r_retail": np.log(m["retail"]).diff(), "r_wholesale": np.log(m["wholesale"]).diff()}).dropna()
        m["iso"] = False
        m.loc[feats.index, "iso"] = IsolationForest(contamination=0.03, random_state=SEED).fit(feats).predict(feats) == -1
        m["flag"] = (m["z"].abs() >= 3.5) | m["iso"]
        seen = m["margin"].notna().to_numpy()
        in_win = np.array([in_events(p) for p in m.index])
        flag = m["flag"].to_numpy()
        stats[key] = {"months": int(seen.sum()), "first": str(m.index.min()), "last": str(m.index.max()),
                      "margin_median_pct": round(float(m["margin"].median() * 100), 2),
                      "margin_range_pct": [round(float(m["margin"].min() * 100), 2), round(float(m["margin"].max() * 100), 2)],
                      "flags": int(flag.sum()), "flags_in_windows": int((flag & in_win).sum()),
                      "months_in_windows": int((seen & in_win).sum())}
        for month, row in m[m["flag"]].iterrows():
            flagged.append({"series": key, "month": str(month), "margin_pct": round(float(row["margin"] * 100), 2),
                            "z": None if np.isnan(row["z"]) else round(float(row["z"]), 2),
                            "isolation_forest": bool(row["iso"]),
                            "reported_window": next((n for n, (a, b) in EVENTS.items()
                                                     if pd.Period(a, "M") <= month <= pd.Period(b, "M")), None)})
    total = sum(s["months"] for s in stats.values())
    flags = sum(s["flags"] for s in stats.values())
    base = sum(s["months_in_windows"] for s in stats.values()) / total
    hit = sum(s["flags_in_windows"] for s in stats.values()) / flags if flags else None
    return {"per_series": stats, "flagged": flagged, "share_of_months_in_windows": round(base, 4),
            "share_of_flags_in_windows": None if hit is None else round(hit, 4),
            "enrichment": None if not hit else round(hit / base, 2),
            "note": "No hoarding labels exist; the windows are publicly reported disruptions, so this is a case study."}


# -------------------------------------------------------------------------------------------------------------- run

def evaluate(rows, pred_return, split):
    """sMAPE, MAE and direction accuracy of predicted log-returns on one split, priced from each origin."""
    sel = (rows["split"] == split).to_numpy()
    last = np.exp(rows["y"].to_numpy()[sel])
    actual = last * np.exp(rows["target"].to_numpy()[sel])
    forecast = last * np.exp(pred_return[sel])
    return {"n": int(sel.sum()), "smape": round(smape(actual, forecast), 3),
            "mae_tk": round(float(np.mean(np.abs(forecast - actual))), 3),
            "direction_accuracy": direction_accuracy(last, actual, forecast)}


def run(csv_path, out_dir, use_rain=True, max_epochs=300, provenance=None, score_test=True):
    """Train every model, choose on validation, score the test year once, forecast ahead, and write the outputs.

    score_test=False leaves the test year unread, for debugging; the recorded run scores it once."""
    import hashlib
    import pathlib
    import time

    import torch

    started = time.time()
    out = pathlib.Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    df = load_wfp(csv_path)
    series, joins = staple_series(df)
    fx = fx_series(df)
    last_month = max(s.index.max() for s in series.values())
    rain = rain_anomaly(df, str(last_month.end_time.date())) if use_rain else {}
    frame = build_frame(series, joins, fx, rain)

    xs, ys, splits, index = sequence_arrays(frame, bool(rain))
    nets = {kind: fit_sequence(kind, xs, ys, splits, max_epochs) for kind in ("gru", "tcn")}
    seq_pred = {kind: dict(zip(index, predict_sequence(net, xs))) for kind, net in nets.items()}
    ylog = {(k, m): v for k, m, v in zip(frame["key"], frame["month"], frame["y"]) if not np.isnan(v)}

    results, boosters = {}, {}
    SPLITS_SCORED = ("val", "test") if score_test else ("val",)
    for h in HORIZONS:
        rows = horizon_rows(frame, h)
        boosters[h] = fit_lightgbm(rows)
        keys, months, y = rows["key"].to_numpy(), rows["month"].to_numpy(), rows["y"].to_numpy()
        preds = {"naive": np.zeros(len(rows)),
                 # the same month a year earlier; with no value there it falls back to the naive forecast
                 "seasonal_naive": np.nan_to_num(np.array([ylog.get((k, m + h - 12), np.nan) for k, m in zip(keys, months)]) - y),
                 "lightgbm": boosters[h].predict(rows[LGB_FEATURES], num_iteration=boosters[h].best_iteration)}
        for kind in nets:
            preds[kind] = np.array([seq_pred[kind].get((k, m), [np.nan] * len(HORIZONS))[h - 1] for k, m in zip(keys, months)])
        preds["ensemble"] = np.nanmean([preds["lightgbm"], preds["gru"], preds["tcn"]], axis=0)
        for kind in ("gru", "tcn", "ensemble"):
            preds[kind] = np.nan_to_num(preds[kind])   # origins without a full 24-month window fall back to naive
        results[h] = {name: {s: evaluate(rows, p, s) for s in SPLITS_SCORED} for name, p in preds.items()}

    names = list(results[HORIZONS[0]])
    mean_smape = {s: {n: round(float(np.mean([results[h][n][s]["smape"] for h in HORIZONS])), 3) for n in names}
                  for s in SPLITS_SCORED}
    chosen = min(names, key=lambda n: mean_smape["val"][n])
    h3 = HORIZONS[-1]
    targets = {} if not score_test else {
        "mean_smape_below_naive": {"chosen": mean_smape["test"][chosen], "naive": mean_smape["test"]["naive"]},
        "h3_smape_below_seasonal_naive": {"chosen": results[h3][chosen]["test"]["smape"],
                                          "seasonal_naive": results[h3]["seasonal_naive"]["test"]["smape"]},
        "h3_direction_accuracy_at_least_0.55": {"chosen": results[h3][chosen]["test"]["direction_accuracy"]},
    }
    if score_test:
        targets["mean_smape_below_naive"]["meets"] = targets["mean_smape_below_naive"]["chosen"] < targets["mean_smape_below_naive"]["naive"]
        targets["h3_smape_below_seasonal_naive"]["meets"] = (targets["h3_smape_below_seasonal_naive"]["chosen"]
                                                             < targets["h3_smape_below_seasonal_naive"]["seasonal_naive"])
        dacc = targets["h3_direction_accuracy_at_least_0.55"]["chosen"]
        targets["h3_direction_accuracy_at_least_0.55"]["meets"] = dacc is not None and dacc >= 0.55

    # Forecasts from each series' latest observed month, with the model chosen on validation.
    latest = frame[frame["y"].notna()].sort_values("month").groupby("key").tail(1)
    forecasts = []
    for _, row in latest.iterrows():
        origin, price = row["month"], float(np.exp(row["y"]))
        s = series[(list(STAPLES)[int(row["staple"])], DIVISIONS[int(row["division"])])]
        ahead = []
        for h in HORIZONS:
            feats = row.copy()
            feats["month_t"], feats["ramadan_t"] = (origin + h).month, ramadan_cached(origin + h)
            lgb_r = float(boosters[h].predict(pd.DataFrame([feats[LGB_FEATURES]]).astype(float),
                                              num_iteration=boosters[h].best_iteration)[0])
            seq_r = {kind: float(seq_pred[kind].get((row["key"], origin), [0.0] * len(HORIZONS))[h - 1]) for kind in nets}
            r = {"naive": 0.0, "seasonal_naive": float(np.nan_to_num(ylog.get((row["key"], origin + h - 12), np.nan) - row["y"])),
                 "lightgbm": lgb_r, **seq_r, "ensemble": float(np.mean([lgb_r, *seq_r.values()]))}[chosen]
            ahead.append({"month": str(origin + h), "price_tk": round(price * float(np.exp(r)), 2)})
        history = s.dropna().iloc[-24:]
        forecasts.append({"key": row["key"], "staple": list(STAPLES)[int(row["staple"])], "division": DIVISIONS[int(row["division"])],
                          "unit": "L" if row["key"].startswith("palm_oil") else "KG", "last_month": str(origin),
                          "last_price_tk": round(price, 2), "model": chosen, "forecast": ahead,
                          "history": [[str(m), round(float(v), 2)] for m, v in history.items()]})

    for h, booster in boosters.items():
        booster.save_model(str(out / f"lightgbm_h{h}.txt"), num_iteration=booster.best_iteration)
    for kind, net in nets.items():
        torch.save(net.state_dict(), out / f"{kind}.pt")
    report = {
        "model": "krishi-price-v1",
        "status": "trained" if score_test else "debug run: validation only, test year not read",
        "data": {"source": "WFP Bangladesh - Food Prices (HDX)", "licence": "CC BY-IGO",
                 "sha256": hashlib.sha256(open(csv_path, "rb").read()).hexdigest(), "rows": int(len(df)),
                 "first": str(df["month"].min()), "last": str(df["month"].max()), **(provenance or {})},
        "series": {f"{STAPLES[c]}|{d}": {"first": str(s.index.min()), "last": str(s.index.max()),
                                          "observed_months": int(s.notna().sum())} for (c, d), s in series.items()},
        "joins": {k: {"first_month": str(v["first_month"]), "log_jump": v["log_jump"]} for k, v in joins.items()},
        "features": {"rainfall": bool(rain), "fx": "taka per USD implied by WFP price / usdprice",
                     "ramadan": "share of the month in Ramadan, tabular Islamic calendar"},
        "windows": {"train": f"targets before {VAL_START}", "val": f"{VAL_START} to {TEST_START - 1}",
                    "test": f"{TEST_START} to {TEST_END}"},
        "training": {"lightgbm_best_iteration": {h: int(b.best_iteration) for h, b in boosters.items()},
                     "gru_epochs": nets["gru"].epochs_run, "tcn_epochs": nets["tcn"].epochs_run,
                     "sequence_windows": int(len(xs))},
        "results": {str(h): results[h] for h in HORIZONS},
        "mean_smape": mean_smape,
        "chosen_on_validation": chosen,
        "targets": targets,
        "spread_anomalies": spread_anomalies(df),
        "seconds": round(time.time() - started, 1),
    }
    (out / "price_eval_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    (out / "price_forecasts.json").write_text(json.dumps({"model": chosen, "made_from_data_through": str(last_month),
                                                          "series": forecasts}, indent=2), encoding="utf-8")
    return report


In [ ]:
CSV = glob.glob("/kaggle/input/**/wfp_food_prices_bgd.csv", recursive=True)[0]
PROVENANCE = {"hdx_dataset": "wfp-food-prices-for-bangladesh", "hdx_last_modified": "2026-11-04T01:13:39"}
report = run(CSV, "/kaggle/working/krishi-price-v1", provenance=PROVENANCE, score_test=True)
print("chosen on validation:", report["chosen_on_validation"], "| rainfall:", report["features"]["rainfall"], "| seconds:", report["seconds"])
for h, by_model in report["results"].items():
    print(f"h={h} test sMAPE", {m: v["test"]["smape"] for m, v in by_model.items()})
    print(f"    direction", {m: v["test"]["direction_accuracy"] for m, v in by_model.items()})
for name, t in report["targets"].items():
    print(f"{name:<38} {t}")
sa = report["spread_anomalies"]
print("spread flags in reported windows:", sa["share_of_flags_in_windows"], "vs months in windows:", sa["share_of_months_in_windows"])
